# Session 11: Filtering, smoothing, baseline correction

A quality-control analyst reports every impurity in a drug-substance chromatogram whose area is at least 0.05% of the main peak's, and on a baseline that rises with the gradient, whether a small peak reaches 0.05% depends on how the baseline was removed. This notebook smooths a chromatogram with a Savitzky-Golay filter, fits and subtracts a polynomial baseline through anchor regions, and reports each impurity's area as a percentage of the main peak. Section 1 works the method on Dataset A and shows what the window and the baseline order each do to the answer. Section 2 applies it to your own record, with a plan from the assistant read against your data before the baseline step. Section 3 compares two baseline orders and asks which the data justify. Section 4 is the judgment cell.

The data are simulated by the course's seeded generator; `data/README.md` describes the method, `chromatogram_a.csv` and `records.csv`, which holds each student's record.

**To submit.** Put your name in the next cell. Click **Restart** in the notebook toolbar, then **Run All**, check that every cell ran without an error, save, and upload `activity.ipynb` to the Session 11 assignment on Canvas by 1:30 pm on Tuesday, November 10. Submissions are individual. Every record is in `data/records.csv`, so the graders' run reads the same data yours did. The notebook is scored 0 to 3: 1 if it runs from top to bottom after a restart and holds your prompt record from section 2, and 0 to 2 for the judgment cell, by the rubric in the syllabus, with the format in `ai-practices.md`.

*Name:*



## Setup and imports

The next cell imports `sys`, `Path` from `pathlib`, NumPy, pandas and Matplotlib, and one new function: `savgol_filter` from `scipy.signal`, the Savitzky-Golay filter, which fits a short polynomial inside a sliding window and keeps its value at the centre. The baseline fit uses `np.polyfit` and `np.polyval`, both in NumPy. The cell prints nothing when it works.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

## Parameters

The two data files and the method's table of known peaks; how far an anchor must sit from a listed peak and how far an integration window reaches; the reporting threshold; the smoothing window and order, and the windows section 1 tries; the baseline order; then your record, set in section 2. Leave `RECORD` as `CHANGE-ME` until section 2.

In [ ]:
DATA_A = Path("data/chromatogram_a.csv")        # Dataset A, section 1's worked example
RECORDS_CSV = Path("data/records.csv")          # every student's record, one column each, sections 2 and 3
TABLE_T_MIN = {"impurity 1": 7.4, "main": 12.0, "impurity 2": 18.6, "impurity 3": 25.2}   # the method's known peaks, min
EXCLUDE_MIN = 1.0           # anchors: every point farther than this from a listed peak, min
HALF_WINDOW_MIN = 0.4       # integration limits each side of a peak's retention time, min
THRESHOLD_PCT = 0.05        # ICH Q3A(R2) reporting threshold, % of the main peak's area, for a daily dose of 2 g or less
SG_WINDOW_POINTS = 11       # section 1's Savitzky-Golay window, points: 2.2 s at 5 points per second
SG_ORDER = 3                # the polynomial fitted inside each window
WINDOWS_TO_TRY = [11, 51, 101, 201]   # section 1 shows what each window does to the main peak
BASELINE_ORDER = 3          # the polynomial order of sections 1 and 2; section 3 sets it beside order 1
ZOOM_MAU = 2.0              # the corrected trace is drawn within this of zero, where the impurities show
RECORD = "CHANGE-ME"        # your assigned record, for example "record_07", section 2

## 1. Explore: a small peak on a rising baseline

Dataset A is one run of the method: a main peak near 1,200 mAU, three known impurities under 1 mAU, and a baseline that climbs about 30 mAU as the gradient's organic fraction rises. The question for this section is what the smoothing window and the baseline's polynomial order each do to an impurity's area percent. The first cell loads the file and plots it twice: at full scale, where only the main peak and the rise show, and zoomed on impurity 1 at 7.4 min, a bump of about a quarter of a milli-absorbance unit on a slope.

In [ ]:
chrom_a = pd.read_csv(DATA_A)
t_min = chrom_a["time_min"].to_numpy()               # min
y_a = chrom_a["absorbance_mAU"].to_numpy()           # mAU, raw
dt_min = t_min[1] - t_min[0]                         # min between points
print(f"{y_a.size} points, {dt_min * 60:.1f} s apart, {t_min[-1]:.0f} min; largest {y_a.max():.0f} mAU at {t_min[np.argmax(y_a)]:.2f} min")
print(f"baseline from about {np.median(y_a[t_min < 1]):.1f} mAU in the first minute to {np.median(y_a[t_min > 29]):.1f} mAU in the last")

near_1 = np.abs(t_min - TABLE_T_MIN["impurity 1"]) < 1.0
fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(10, 4))
ax_full.plot(t_min, y_a, color="black", linewidth=0.8)
ax_full.set_xlabel("retention time (min)")
ax_full.set_ylabel("absorbance (mAU)")
ax_full.set_title("Dataset A, full scale")
ax_zoom.plot(t_min[near_1], y_a[near_1], color="black", linewidth=0.8)
ax_zoom.set_xlabel("retention time (min)")
ax_zoom.set_ylabel("absorbance (mAU)")
ax_zoom.set_title("Dataset A near impurity 1 (7.4 min)")
fig.tight_layout()
plt.show()

At full scale the trace is one 1220 mAU peak at 12.00 min on a baseline that rises from about 3.7 mAU in the first minute to 44.4 mAU in the last; zoomed near 7.4 min, impurity 1 is a bump of about 0.8 mAU, from about 13.3 to 14.2 mAU, on a slope of about 1.4 mAU per minute, with noise of a few hundredths of a milli-absorbance unit on every point.

Noise of about 0.03 mAU sits on every point, and a peak a quarter of a milli-absorbance unit tall needs it reduced before its shape can be read. A Savitzky-Golay filter fits a polynomial of order `SG_ORDER` to the points inside a sliding window and keeps the fit's value at the centre, so it averages noise away while following a curve. The window must be narrower than the narrowest peak that has to survive: here every peak is about 43 to 64 points wide at half height. The next cell tries four windows and prints the main peak's height after each against its raw height, and the residual, raw minus smoothed, inside the main peak against outside it.

In [ ]:
def peak_height(t, y, t_peak, half_window):
    """The largest value of y within half_window of t_peak, in y's units."""
    return y[np.abs(t - t_peak) <= half_window].max()


inside_main = np.abs(t_min - TABLE_T_MIN["main"]) <= HALF_WINDOW_MIN / 2
outside_main = (np.abs(t_min - TABLE_T_MIN["main"]) > HALF_WINDOW_MIN) & (np.abs(t_min - TABLE_T_MIN["main"]) < 1.0)
height_raw_a = peak_height(t_min, y_a, TABLE_T_MIN["main"], HALF_WINDOW_MIN)
rows = []
for window_points in WINDOWS_TO_TRY:
    smoothed = savgol_filter(y_a, window_points, SG_ORDER)
    residual = y_a - smoothed
    rows.append({"window_points": window_points, "window_s": window_points * dt_min * 60,
                 "main_height_mAU": peak_height(t_min, smoothed, TABLE_T_MIN["main"], HALF_WINDOW_MIN),
                 "height_kept": peak_height(t_min, smoothed, TABLE_T_MIN["main"], HALF_WINDOW_MIN) / height_raw_a,
                 "residual_inside_mAU": residual[inside_main].std(), "residual_outside_mAU": residual[outside_main].std()})
windows_a = pd.DataFrame(rows)
print(f"main peak raw: {height_raw_a:.0f} mAU")
print(windows_a.round(3).to_string(index=False))

y_a_s = savgol_filter(y_a, SG_WINDOW_POINTS, SG_ORDER)   # the smoothed trace the rest of the section uses

The baseline is fitted only through **anchors**, points where no peak is: every point more than `EXCLUDE_MIN` from each listed retention time. `np.polyfit` fits a polynomial of the chosen order through the anchors on the raw trace, and `np.polyval` evaluates it everywhere, under the peaks too. Subtracted from the smoothed trace, it leaves the **corrected trace**, at zero between peaks if it is right. The check is the **anchor residual**, the corrected trace inside the anchors: a bow means the order is too low, a bump means a peak sits in an anchor. The next cell fits order 1 and 3, prints each residual's largest value and where, and draws the corrected trace at ±2 mAU, then both residuals.

In [ ]:
def anchor_mask(t, excluded, half_width):
    """True at every point farther than half_width from each retention time in excluded."""
    mask = np.ones_like(t, dtype=bool)
    for t_peak in excluded:
        mask &= np.abs(t - t_peak) > half_width
    return mask


def fit_baseline(t, y, mask, order):
    """A polynomial of the given order through the points where mask is True, evaluated at every t."""
    return np.polyval(np.polyfit(t[mask], y[mask], order), t)


def worst_residual(t, y_corr, mask):
    """The largest absolute value of the corrected trace inside the anchors, and the time it sits at."""
    i = np.flatnonzero(mask)[np.argmax(np.abs(y_corr[mask]))]
    return abs(y_corr[i]), t[i]


mask_a = anchor_mask(t_min, TABLE_T_MIN.values(), EXCLUDE_MIN)
corrected_a = {}
for order in (1, BASELINE_ORDER):
    corrected_a[order] = y_a_s - fit_baseline(t_min, y_a, mask_a, order)
    worst, at = worst_residual(t_min, corrected_a[order], mask_a)
    print(f"order {order}: {mask_a.sum()} anchor points; largest anchor residual {worst:.3f} mAU at {at:.2f} min")
corr_a = corrected_a[BASELINE_ORDER]

fig, (ax_trace, ax_resid) = plt.subplots(2, 1, figsize=(8, 6))
ax_trace.plot(t_min, corr_a, color="black", linewidth=0.8)
for name, t_peak in TABLE_T_MIN.items():
    ax_trace.axvline(t_peak, color="gray", linestyle=":", linewidth=0.8)
ax_trace.set_ylim(-ZOOM_MAU, ZOOM_MAU)
ax_trace.set_xlabel("retention time (min)")
ax_trace.set_ylabel("corrected absorbance (mAU)")
ax_trace.set_title(f"Dataset A, baseline order {BASELINE_ORDER} removed; dotted lines at the listed peaks")
ax_resid.plot(t_min, np.where(mask_a, corrected_a[1], np.nan), color="gray", linestyle="-", linewidth=0.8, label="order 1")
ax_resid.plot(t_min, np.where(mask_a, corr_a, np.nan), color="black", linestyle="--", linewidth=0.8, label=f"order {BASELINE_ORDER}")
ax_resid.axhline(0, color="black", linewidth=0.5)
ax_resid.set_xlabel("retention time (min)")
ax_resid.set_ylabel("anchor residual (mAU)")
ax_resid.set_title("Dataset A: the corrected trace inside the anchors")
ax_resid.legend()
fig.tight_layout()
plt.show()

With the order 3 baseline removed the corrected trace sits at zero between peaks, with impurity 1 at about 0.8 mAU, impurity 2 at 0.3 mAU and impurity 3 at 0.6 mAU above it at the dotted lines and the main peak off the scale; inside the anchors the order 1 residual bows from about -0.06 mAU near 10 min to +0.07 mAU between 15 and 20 min and -0.14 mAU at the end of the run, while the order 3 residual stays within 0.063 mAU of zero with no long run of one sign.

A peak's area is the corrected trace summed over its integration window, `HALF_WINDOW_MIN` each side of its listed time, times the spacing of the points in minutes: mAU times min. Each impurity's area divided by the main peak's, times 100, is its area percent, which the analyst sets against the 0.05% reporting threshold. The next cell computes every peak's area percent under both baseline orders and prints the table with the reporting call each order gives.

In [ ]:
def area(t, y_corr, t_peak, half_window, dt):
    """The corrected trace summed within half_window of t_peak, times the point spacing: mAU min."""
    return y_corr[np.abs(t - t_peak) <= half_window].sum() * dt


rows = []
for order, y_corr in corrected_a.items():
    main_area = area(t_min, y_corr, TABLE_T_MIN["main"], HALF_WINDOW_MIN, dt_min)
    for name, t_peak in TABLE_T_MIN.items():
        if name != "main":
            pct = area(t_min, y_corr, t_peak, HALF_WINDOW_MIN, dt_min) / main_area * 100
            rows.append({"peak": name, "t_min": t_peak, "order": order, "main_area_mAU_min": main_area,
                         "area_pct": pct, "reported": bool(pct >= THRESHOLD_PCT)})
areas_a = pd.DataFrame(rows)
print(f"0.05% of the main peak's area is {areas_a['main_area_mAU_min'].iloc[0] * THRESHOLD_PCT / 100:.4f} mAU min")
print(areas_a.pivot(index="peak", columns="order", values="area_pct").round(4).to_string())
for order in corrected_a:
    reported = areas_a[(areas_a["order"] == order) & areas_a["reported"]]["peak"].tolist()
    print(f"order {order} reports: {', '.join(reported) if reported else 'none'}")

**Explain.** In two sentences: impurity 2 is reported under order 1 and not under order 3. What in the two residual plots shows which order to trust, and what does the difference in its area percent come from?

*Your answer:*

## 2. Apply: your record, and a plan read against it

Your record is another run of the same method, whose impurity levels, baseline and noise differ from Dataset A's and from your neighbor's. The question for your record: which impurities reach 0.05% of the main peak, once the baseline is removed through anchors that hold no peak. You choose the smoothing window, look at your corrected record, ask the assistant for a plan for the baseline step and read it against what you saw, then write the anchor cell yourself.

**[TASK 1]** Set `RECORD` in the Parameters cell to your assigned record, run the Parameters cell, then run the next cell. It reads your column from `data/records.csv` and checks that its time axis is Dataset A's. Until `RECORD` is set, it stops with `KeyError: 'CHANGE-ME'`.

In [ ]:
records = pd.read_csv(RECORDS_CSV)
y_b = records[RECORD].to_numpy()                      # mAU, raw: KeyError until RECORD names a column
assert np.allclose(records["time_min"].to_numpy(), t_min), "the records' time axis differs from Dataset A's"
print(f"{RECORD}: {y_b.size} points, {dt_min * 60:.1f} s apart; largest {y_b.max():.0f} mAU at {t_min[np.argmax(y_b)]:.2f} min")
print(f"baseline from about {np.median(y_b[t_min < 1]):.1f} mAU in the first minute to {np.median(y_b[t_min > 29]):.1f} mAU in the last")

**[TASK 2]** Choose the smoothing window for your record: an odd number of points, read from section 1's table and the width of the peaks, about 43 to 64 points at half height. The cell smooths your record with it, and prints the main peak's height raw and smoothed and the residual inside the main peak against outside it, the same check as section 1. A window that keeps the main peak's height within about 1% kept the peaks; the residual inside the main peak is always larger than the noise outside it and grows with the window, from 0.05 mAU at 11 points to 16 mAU at 51 on Dataset A. Until it is filled, the cell stops with `TypeError: int() argument must be a string, a bytes-like object or a real number, not 'ellipsis'`.

In [ ]:
window_points = int(...)   # [TASK 2] an odd number of points; section 1's table says what each does to the main peak
y_b_s = savgol_filter(y_b, window_points, SG_ORDER)
residual_b = y_b - y_b_s
height_raw_b = peak_height(t_min, y_b, TABLE_T_MIN["main"], HALF_WINDOW_MIN)
height_smoothed_b = peak_height(t_min, y_b_s, TABLE_T_MIN["main"], HALF_WINDOW_MIN)
print(f"{RECORD}, window {window_points} points ({window_points * dt_min * 60:.1f} s): main peak {height_raw_b:.0f} mAU raw, "
      f"{height_smoothed_b:.0f} mAU smoothed, {height_smoothed_b / height_raw_b:.0%} kept")
print(f"residual rms inside the main peak {residual_b[inside_main].std():.3f} mAU, outside {residual_b[outside_main].std():.3f} mAU")

**Explore and plan before code**, today's AI practice, met in session 7, before you look further at your record. Save the notebook (**File**, then **Save**), so the assistant reads the record you set and TASK 2's output. In Claude Code, click the mode indicator at the bottom of the prompt box and choose **Plan**, or type `/plan` and a space before your prompt; if neither is there, type the prompt without it, since the course's `CLAUDE.md` asks for a plan in words anyway.

Then ask, in your own words, for a plan for the baseline step on your record in `activity.ipynb`: which points anchor the baseline, the polynomial order, the smoothing window and the order of the steps, in words and with no code until you approve. Do not approve it yet. Add a markdown cell under this one (hover just below this cell and click **+ Markdown**) holding these lines, and write your prompt on the first:

```
**Prompt record**

- **My prompt:**
- **What the plan proposed for the anchors, and what the first look showed about them:**
- **What I changed before approving, or what I verified:**
```

The next cell is a first look at your record with section 1's anchors. Read the plan's anchor sentence against it: does it place anchors where your record has a peak, listed or not, and does it say what to do about a peak the table does not list? A plan built from the method's table can be right for the table and wrong for your record. Fill the record's second and third lines after the first look; TASK 3 is where a change goes. One prompt that works, for after you have sent your own: `/plan Remove the gradient baseline from my record in activity.ipynb and report each impurity's area as a percentage of the main peak. Say in words which points anchor the baseline, the polynomial order, the Savitzky-Golay window and the order of the steps. No code until I approve.` Your prompt record is one candidate for part (b) of the judgment cell.

A first look at your record with section 1's anchors, every point more than `EXCLUDE_MIN` from the four listed peaks, and section 1's order: the baseline a plan built from the table draws. The next cell fits it, subtracts it, prints the largest anchor residual and where it sits beside Dataset A's and each listed impurity's area percent through those anchors, and draws your corrected record at ±2 mAU with the listed times marked. Look at the whole run, not only at the marked times. Anything the method's table does not list is yours to find.

In [ ]:
mask_table = anchor_mask(t_min, TABLE_T_MIN.values(), EXCLUDE_MIN)
corr_table = y_b_s - fit_baseline(t_min, y_b, mask_table, BASELINE_ORDER)
worst_table, at_table = worst_residual(t_min, corr_table, mask_table)
worst_a, at_a = worst_residual(t_min, corr_a, mask_a)
print(f"{RECORD}, section 1's anchors, order {BASELINE_ORDER}: largest anchor residual {worst_table:.2f} mAU at {at_table:.2f} min; "
      f"Dataset A's was {worst_a:.3f} mAU at {at_a:.2f} min")
main_area_table = area(t_min, corr_table, TABLE_T_MIN["main"], HALF_WINDOW_MIN, dt_min)
pct_table = {name: area(t_min, corr_table, t_peak, HALF_WINDOW_MIN, dt_min) / main_area_table * 100
             for name, t_peak in TABLE_T_MIN.items() if name != "main"}
print("area percent through section 1's anchors: " + ", ".join(f"{name} {pct:.4f}%" for name, pct in pct_table.items()))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(t_min, corr_table, color="black", linewidth=0.8)
for name, t_peak in TABLE_T_MIN.items():
    ax.axvline(t_peak, color="gray", linestyle=":", linewidth=0.8)
ax.set_ylim(-ZOOM_MAU, ZOOM_MAU)
ax.set_xlabel("retention time (min)")
ax.set_ylabel("corrected absorbance (mAU)")
ax.set_title(f"{RECORD} with section 1's anchors; dotted lines at the listed peaks")
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here.*

**[TASK 3]** Write the anchor cell for your record yourself, in a code cell you add under this one (**+ Code**), by the method section 1 used for Dataset A, after reading the plan against the first look. It sets two names. `extra_t_min` is the retention time, in minutes, of a peak your record shows that the method's table does not list, read from the plot and the printout above, or `None` if there is none. `anchors` is the boolean mask from `anchor_mask` that excludes `EXCLUDE_MIN` on each side of every listed peak and of `extra_t_min`. The cell after it fits the baseline through your anchors, prints the largest anchor residual and every peak's area percent beside the first look's, and stops with `NameError: name 'anchors' is not defined` until your cell sets it.

In [ ]:
corr_b = y_b_s - fit_baseline(t_min, y_b, anchors, BASELINE_ORDER)   # set by the code cell you added above
worst_b, at_b = worst_residual(t_min, corr_b, anchors)
peaks_b = dict(TABLE_T_MIN)
if extra_t_min is not None:
    peaks_b["unlisted"] = extra_t_min
main_area_b = area(t_min, corr_b, TABLE_T_MIN["main"], HALF_WINDOW_MIN, dt_min)
rows = []
for name, t_peak in peaks_b.items():
    if name != "main":
        pct = area(t_min, corr_b, t_peak, HALF_WINDOW_MIN, dt_min) / main_area_b * 100
        rows.append({"peak": name, "t_min": t_peak, "area_pct_section_1": pct_table.get(name), "area_pct": pct,
                     "reported": bool(pct >= THRESHOLD_PCT)})
areas_b = pd.DataFrame(rows)
print(f"{RECORD}, your anchors, order {BASELINE_ORDER}: {anchors.sum()} anchor points, largest anchor residual {worst_b:.3f} mAU at {at_b:.2f} min, "
      f"against {worst_table:.2f} mAU with section 1's anchors")
print(f"main peak {main_area_b:.1f} mAU min; 0.05% of it {main_area_b * THRESHOLD_PCT / 100:.4f} mAU min")
print(areas_b.round(4).to_string(index=False))

**Explain.** Before you compare: in two sentences, give the time and area percent of any peak your record showed that the table does not list, and which listed impurity's area percent moved most between the first look's anchors and yours, by how much and in which direction. Then compare with a neighbor: would section 1's anchors have changed a reporting call on their record, and by how much? Add one sentence on why a baseline fitted through a peak moves its neighbors, and in which direction.

*Your answer:*

## 3. Analyze: choose the baseline order from the residual

Two baselines through your anchors: treatment A, a straight line (order 1), and treatment B, order 3, as in sections 1 and 2. Both are defensible on a baseline that looks nearly straight at full scale, and they differ by a fraction of a milli-absorbance unit, which is the size of the impurities. The next cell fits both, prints every peak's area percent under each with the reporting call each gives, and the largest anchor residual under each, then draws the corrected trace around the listed impurity whose area percent moves most between the orders, and both anchor residuals.

In [ ]:
rows, corrected_b, worst_by_order = [], {}, {}
for order in (1, BASELINE_ORDER):
    corrected_b[order] = y_b_s - fit_baseline(t_min, y_b, anchors, order)
    worst_by_order[order] = worst_residual(t_min, corrected_b[order], anchors)
    main_area_order = area(t_min, corrected_b[order], TABLE_T_MIN["main"], HALF_WINDOW_MIN, dt_min)
    for name, t_peak in peaks_b.items():
        if name != "main":
            pct = area(t_min, corrected_b[order], t_peak, HALF_WINDOW_MIN, dt_min) / main_area_order * 100
            rows.append({"peak": name, "t_min": t_peak, "order": order, "area_pct": pct, "reported": bool(pct >= THRESHOLD_PCT)})
orders_b = pd.DataFrame(rows)
by_order = orders_b.pivot(index="peak", columns="order", values="area_pct")
by_order["difference"] = by_order[BASELINE_ORDER] - by_order[1]
print(by_order.round(4).to_string())
for order in corrected_b:
    reported = orders_b[(orders_b["order"] == order) & orders_b["reported"]]["peak"].tolist()
    print(f"order {order} reports: {', '.join(reported) if reported else 'none'}; largest anchor residual "
          f"{worst_by_order[order][0]:.3f} mAU at {worst_by_order[order][1]:.2f} min")
listed = by_order.drop(index="unlisted", errors="ignore")
moves_most = listed["difference"].abs().idxmax()
t_moves = peaks_b[moves_most]
print(f"{moves_most} moves most between the orders: {by_order.loc[moves_most, 1]:.4f}% under order 1, {by_order.loc[moves_most, BASELINE_ORDER]:.4f}% under order {BASELINE_ORDER}")
reported_by_order = orders_b.pivot(index="peak", columns="order", values="reported")
differ = [name for name in peaks_b if name != "main" and reported_by_order.loc[name, 1] != reported_by_order.loc[name, BASELINE_ORDER]]
print(f"calls that differ between the orders: {', '.join(differ) if differ else 'none'}")

near = np.abs(t_min - t_moves) < 1.5 * HALF_WINDOW_MIN
fig, (ax_peak, ax_resid) = plt.subplots(1, 2, figsize=(10, 4))
ax_peak.plot(t_min[near], corrected_b[1][near], color="gray", linestyle="-", linewidth=0.8, label="A, order 1")
ax_peak.plot(t_min[near], corrected_b[BASELINE_ORDER][near], color="black", linestyle="--", linewidth=0.8, label=f"B, order {BASELINE_ORDER}")
ax_peak.axhline(0, color="black", linewidth=0.5)
ax_peak.set_xlabel("retention time (min)")
ax_peak.set_ylabel("corrected absorbance (mAU)")
ax_peak.set_title(f"{RECORD}: {moves_most} under each baseline")
ax_peak.legend()
ax_resid.plot(t_min, np.where(anchors, corrected_b[1], np.nan), color="gray", linestyle="-", linewidth=0.8, label="A, order 1")
ax_resid.plot(t_min, np.where(anchors, corrected_b[BASELINE_ORDER], np.nan), color="black", linestyle="--", linewidth=0.8, label=f"B, order {BASELINE_ORDER}")
ax_resid.axhline(0, color="black", linewidth=0.5)
ax_resid.set_xlabel("retention time (min)")
ax_resid.set_ylabel("anchor residual (mAU)")
ax_resid.set_title(f"{RECORD}: the anchor residual under each baseline")
ax_resid.legend()
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here.*

**Explain.** In two sentences, say which order the anchor residual justifies and why, citing both residuals' largest values, and which impurity's reporting call, or calls, the choice changes and what that means in the batch record. This is the evidence part (a) of the judgment cell needs.

*Your answer:*

## 4. Judgment cell

The cell below is scored 0 to 2 by the rubric in the syllabus, in the format that `ai-practices.md` gives, and you write it yourself. Part (a): as the analyst, which peaks of your record go into the batch record as impurities at or above 0.05%, and at what area percent? Cite the baseline order you chose and its largest anchor residual against the other's, the impurity whose call the order changes, and the level of any peak the method's table does not list, and say what a wrong call would cost either way. Part (b): one judgment entry, a suggestion from the assistant that you changed, rejected or verified, and the reason from your data. Your prompt record from section 2 is the natural source; the reason has to cite a value from your output, such as the anchor residual with the plan's anchors against yours, not a general rule. This session's AI practice, named in the notes, is *explore and plan before code*, and an entry that came from using it is the expected one, though any entry grounded in your data counts.

**Judgment cell**

**(a)**

**(b) Judgment entry**

- **What the assistant suggested:**
- **What I did:**
- **Why, from my data:** 

## Software environment

The last cell prints the Python and package versions this notebook ran with, so a result can always be traced to the software that produced it.

In [ ]:
import matplotlib
import scipy

print(f"Python     {sys.version.split()[0]}")
print(f"NumPy      {np.__version__}")
print(f"pandas     {pd.__version__}")
print(f"SciPy      {scipy.__version__}")
print(f"Matplotlib {matplotlib.__version__}")